In [1]:
# ============================================================
# COLAB USERS ONLY — Uncomment and run this cell if using Google Colab
# If you're running locally in VS Code, SKIP this cell entirely
# ============================================================

# !pip install langchain-groq langchain-google-genai langchain-community smolagents ddgs tiktoken python-dotenv -q

# from google.colab import userdata
# import os
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
# print("Colab setup complete!")

# Day 1: LLM APIs + Your First AI Agent 🚀

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**What you'll build today:**
1. Call an LLM API from Python (Groq + Gemini)
2. Stream responses in real-time
3. Build a reusable LLM wrapper class
4. Create your first AI agent that searches the web and reasons about results

**Time:** 1.5 hours | **Difficulty:** Beginner-friendly | **Prerequisites:** Python basics + setup complete

---

## 0. Setup — Load your API keys

Run this cell first. It loads your keys from the `.env` file you created during pre-class setup.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# Verify keys are loaded
groq_key = os.getenv("GROQ_API_KEY", "")
gemini_key = os.getenv("GOOGLE_API_KEY", "")
hf_token = os.getenv("HF_TOKEN", "")

print(f"Groq API Key:    {'✅ Loaded' if len(groq_key) > 10 else '❌ Missing — check .env file'}")
print(f"Gemini API Key:  {'✅ Loaded' if len(gemini_key) > 10 else '⚠️ Optional — not critical for today'}")
print(f"HF Token:        {'✅ Loaded' if len(hf_token) > 10 else '⚠️ Optional — needed for Section 5'}")

Groq API Key:    ✅ Loaded
Gemini API Key:  ✅ Loaded
HF Token:        ✅ Loaded


---
## 1. 🎯 Your first LLM API call in 5 lines

Forget ChatGPT's web interface. Today you become the person who **builds** things like ChatGPT.

The entire magic of ChatGPT, Gemini, Claude — it's just an **HTTP API call**. Let's make one right now:

In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

response = llm.invoke("What is Agentic AI in one sentence?")
print(response.content)

Agentic AI refers to artificial intelligence systems that are capable of autonomous decision-making, self-directed action, and goal-oriented behavior, allowing them to act independently and make choices that achieve their objectives.


**That's it.** Three lines of code, and you just talked to one of the most powerful AI models in the world.

Now let's understand what actually happened behind the scenes.

---

## 2. How LLM APIs actually work (10 min concept)

When you type in ChatGPT, here's what happens under the hood:

```
Your text → [Tokenizer] → Tokens → [LLM Model] → Output Tokens → [Detokenizer] → Response text
```

**Key concepts you need to know:**

| Concept | What it means | Why it matters |
|---------|--------------|----------------|
| **Tokens** | Words/subwords the model processes (~4 chars = 1 token) | You pay per token. "Hello world" = 2 tokens |
| **Model** | Which AI brain to use (Llama, Gemini, GPT) | Bigger = smarter but slower & costlier |
| **Temperature** | Randomness (0 = deterministic, 1 = creative) | 0 for facts, 0.7 for creative writing |
| **System prompt** | Hidden instructions that shape behavior | Makes the AI act as a specific persona |
| **Max tokens** | Limit on response length | Prevents runaway responses and controls cost |

**The API call structure (every LLM API follows this pattern):**
```
POST /chat/completions
{
  "model": "llama-3.3-70b-versatile",
  "messages": [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is Agentic AI?"}
  ],
  "temperature": 0.7,
  "max_tokens": 500
}
```

Let's explore each of these hands-on.

---
## 3. Groq API — Deep dive (Live coding)

### 3.1 The raw HTTP call (understanding what frameworks hide from you)

Before using any framework, let's see the **raw API call**. This is what LangChain does under the hood:

In [4]:
import requests
import json

url = "https://api.groq.com/openai/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {os.getenv('GROQ_API_KEY')}",
    "Content-Type": "application/json"
}

payload = {
    "model": "llama-3.3-70b-versatile",
    "messages": [
        {"role": "system", "content": "You are a helpful AI assistant. Be concise."},
        {"role": "user", "content": "Explain what an AI agent is in 2 sentences."}
    ],
    "temperature": 0,
    "max_tokens": 200
}

response = requests.post(url, headers=headers, json=payload)
data = response.json()

# The response
print("Response:", data["choices"][0]["message"]["content"])
print()
# The metadata — this is what most people never look at
print(f"Model used:      {data['model']}")
print(f"Tokens (prompt):  {data['usage']['prompt_tokens']}")
print(f"Tokens (output):  {data['usage']['completion_tokens']}")
print(f"Tokens (total):   {data['usage']['total_tokens']}")

Response: An AI agent is a computer program that uses artificial intelligence to perform tasks autonomously, making decisions and taking actions based on its programming and the data it receives. AI agents can be designed to interact with their environment, learn from experience, and adapt to new situations, allowing them to accomplish specific goals or tasks with varying degrees of autonomy.

Model used:      llama-3.3-70b-versatile
Tokens (prompt):  57
Tokens (output):  68
Tokens (total):   125


**Key takeaway:** Every LLM interaction is just a POST request with JSON. Frameworks like LangChain just make this easier. But knowing the raw call helps you debug, optimize costs, and switch providers.

### 3.2 Using LangChain's ChatGroq (the clean way)

Same thing, but with LangChain — cleaner, more Pythonic:

In [5]:
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

# Create the LLM client
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=300,
)

# Send messages (same structure as the raw API call)
messages = [
    SystemMessage(content="You are a senior AI engineer. Be technical and precise."),
    HumanMessage(content="What are the 3 key components that make an AI agent different from a chatbot?")
]

response = llm.invoke(messages)
print(response.content)

The three key components that differentiate an AI agent from a chatbot are:

1. **Autonomy**: AI agents possess the ability to operate independently, making decisions and taking actions without explicit human intervention. They can perceive their environment, reason about the current state, and adapt to changes. In contrast, chatbots typically rely on pre-defined rules and scripts to respond to user input.

2. **Reasoning and Problem-Solving**: AI agents are equipped with advanced reasoning and problem-solving capabilities, enabling them to analyze complex situations, identify patterns, and generate novel solutions. They can learn from experience, update their knowledge, and improve their performance over time. Chatbots, on the other hand, usually rely on pre-defined knowledge bases and lack the ability to reason and solve problems in a more general sense.

3. **Goal-Oriented Behavior**: AI agents are designed to pursue specific goals and objectives, which guide their decision-making a

### 3.3 Temperature experiment — see the difference yourself

Let's call the same prompt with different temperatures and compare:

In [6]:
prompt = "Give me a creative name for an AI startup that builds agents."

for temp in [0, 0.5, 1.0]:
    llm_temp = ChatGroq(model="llama-3.3-70b-versatile", temperature=temp, max_tokens=50)
    response = llm_temp.invoke(prompt)
    print(f"Temperature {temp}: {response.content.strip()}")
    print()

Temperature 0: Here are some creative name suggestions for an AI startup that builds agents:

1. **Agentia**: Derived from the word "agent," this name suggests a company that specializes in creating intelligent agents.
2. **SynapseAI**: This name plays on

Temperature 0.5: Here are some creative name suggestions for an AI startup that builds agents:

1. **Agentia**: Derived from the word "agent," this name suggests a company that specializes in creating intelligent agents.
2. **SynapseAI**: This name plays on

Temperature 1.0: Here are some creative name suggestions for an AI startup that builds agents:

1. **Agentia**: A combination of "agent" and the Latin suffix "-ia," suggesting a place or territory, implying a comprehensive platform for AI agents.
2. **



Notice how temperature=0 gives the same answer every time, while temperature=1.0 gets increasingly creative (and sometimes weird).

### 3.4 Streaming — watch the AI think in real-time

This is how ChatGPT shows text appearing word by word:

In [7]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)

print("Streaming response:")
print("-" * 40)

for chunk in llm.stream("Explain the ReAct pattern in AI agents in 3 bullet points."):
    print(chunk.content, end="", flush=True)

print()
print("-" * 40)
print("Done!")

Streaming response:
----------------------------------------
The ReAct pattern is a design pattern used in AI agents to manage their behavior and decision-making processes. Here are three key points about the ReAct pattern:

* **Separation of Concerns**: The ReAct pattern separates an AI agent's concerns into three main components: Reasoning, Acting, and perceiving the environment (ReAct). This separation allows for a more modular and maintainable architecture, making it easier to develop and update AI agents.
* **Event-Driven Architecture**: The ReAct pattern is based on an event-driven architecture, where the AI agent reacts to events or changes in its environment. The agent perceives its environment, reasons about the current situation, and then acts accordingly. This approach enables the agent to respond to dynamic and unpredictable environments.
* **Decoupling of Reasoning and Acting**: The ReAct pattern decouples the reasoning and acting components, allowing them to operate indep

---
## ✅ Quick check — Predict the output

Before running the cell below, **predict**: What will the token count be for the prompt "Hi"?

Think: "Hi" is 1 word. But is it 1 token?

In [8]:
import tiktoken

encoder = tiktoken.get_encoding("cl100k_base")

test_strings = [
    ("English", "Hi"),
    ("English", "Agentic AI is the future of software engineering."),
    ("Hindi", "नमस्ते"),
    ("Telugu", "నమస్కారం"),
    ("Tamil", "வணக்கம்"),
    ("Malayalam", "നമസ്കാരം"),
    ("Kannada", "ನಮಸ್ಕಾರ"),
    ("Bengali", "নমস্কার"),
    ("Marathi", "नमस्कार"),
]

for lang, s in test_strings:
    tokens = encoder.encode(s)
    print(f'{lang} -- "{s}" --> {len(tokens)} tokens')

print()
print("---")
print("'Hi' = 1 token, but the SAME greeting in Indian languages = 6-16 tokens!")
print("This is why non-English AI costs more per word -- and why multilingual AI is hard.")

English -- "Hi" --> 1 tokens
English -- "Agentic AI is the future of software engineering." --> 10 tokens
Hindi -- "नमस्ते" --> 6 tokens
Telugu -- "నమస్కారం" --> 16 tokens
Tamil -- "வணக்கம்" --> 11 tokens
Malayalam -- "നമസ്കാരം" --> 14 tokens
Kannada -- "ನಮಸ್ಕಾರ" --> 14 tokens
Bengali -- "নমস্কার" --> 9 tokens
Marathi -- "नमस्कार" --> 7 tokens

---
'Hi' = 1 token, but the SAME greeting in Indian languages = 6-16 tokens!
This is why non-English AI costs more per word -- and why multilingual AI is hard.


**Surprise:** Even "Hi" can be 1 token, but non-English text uses many more tokens per word. This matters for cost optimization later in the course.

---

## 4. Guided practice — Google Gemini API + Multi-LLM wrapper (We do together)

### 4.1 Calling Gemini — same pattern, different provider

See how easy it is to switch LLMs? The message structure is identical:

In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI

# If you have GOOGLE_API_KEY set, this will work
# If not, skip this cell — Groq is our primary LLM
try:
    gemini = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash-lite",
        temperature=0,
        max_tokens=200,
    )
    
    response = gemini.invoke("What is Agentic AI in one sentence?")
    print(f"Gemini says: {response.content}")
except Exception as e:
    print(f"Gemini not available: {e}")
    print("No worries — Groq is our primary LLM. Gemini is just a backup.")

Gemini says: Agentic AI refers to artificial intelligence systems that can autonomously perceive their environment, make decisions, and take actions to achieve specific goals.


### 4.2 Build a reusable LLM wrapper — your first utility class

In real projects, you don't want to hardcode the LLM provider everywhere. Let's build a wrapper that lets you switch between Groq, Gemini, or any provider with one parameter:

In [10]:
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

class AgentLLM:
    """Reusable LLM wrapper for the Agentic AI course.
    Supports multiple providers with a single interface.
    """
    
    def __init__(self, provider="groq", model=None, temperature=0, max_tokens=500):
        self.provider = provider
        
        if provider == "groq":
            self.llm = ChatGroq(
                model=model or "llama-3.3-70b-versatile",
                temperature=temperature,
                max_tokens=max_tokens,
            )
        elif provider == "gemini":
            from langchain_google_genai import ChatGoogleGenerativeAI
            self.llm = ChatGoogleGenerativeAI(
                model=model or "gemini-2.5-flash-lite",
                temperature=temperature,
                max_tokens=max_tokens,
            )
        else:
            raise ValueError(f"Unknown provider: {provider}. Use 'groq' or 'gemini'.")
    
    def ask(self, question, system_prompt=None):
        """Simple question → answer."""
        messages = []
        if system_prompt:
            messages.append(SystemMessage(content=system_prompt))
        messages.append(HumanMessage(content=question))
        return self.llm.invoke(messages).content
    
    def stream(self, question, system_prompt=None):
        """Stream response token by token."""
        messages = []
        if system_prompt:
            messages.append(SystemMessage(content=system_prompt))
        messages.append(HumanMessage(content=question))
        for chunk in self.llm.stream(messages):
            print(chunk.content, end="", flush=True)
        print()
    
    def chat(self, messages):
        """Multi-turn conversation with message history."""
        return self.llm.invoke(messages).content


# Test it
ai = AgentLLM(provider="groq", temperature=0.3)

print("=== Simple question ===")
print(ai.ask("What makes an AI agent autonomous?"))
print()

print("=== With system prompt ===")
print(ai.ask(
    "Explain RAG in simple terms",
    system_prompt="You are explaining to a college student. Use analogies. Max 3 sentences."
))

=== Simple question ===
An AI agent is considered autonomous if it can perform tasks and make decisions without human intervention or external control. The key characteristics that make an AI agent autonomous are:

1. **Self-governance**: The agent can operate independently, making decisions based on its own goals, objectives, and rules.
2. **Decision-making**: The agent can analyze situations, weigh options, and choose the best course of action without human input.
3. **Learning and adaptation**: The agent can learn from experience, adapt to new situations, and improve its performance over time.
4. **Perception and sensing**: The agent can perceive its environment, gather data, and use this information to inform its decisions.
5. **Action and execution**: The agent can execute actions and interact with its environment to achieve its goals.
6. **Autonomy levels**: The agent can operate at various levels of autonomy, ranging from fully autonomous (no human intervention) to semi-autonomo

### 4.3 Multi-turn conversation — the AI remembers context

This is the difference between a single API call and a chatbot:

In [11]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

ai = AgentLLM(provider="groq", temperature=0)

# Simulate a conversation with history
conversation = [
    SystemMessage(content="You are a helpful AI tutor. Be concise."),
    HumanMessage(content="What is an embedding?"),
]

# Turn 1
response1 = ai.chat(conversation)
print(f"AI: {response1}")
conversation.append(AIMessage(content=response1))

# Turn 2 — the AI should remember the context
conversation.append(HumanMessage(content="How is it used in RAG?"))
response2 = ai.chat(conversation)
print(f"\nAI: {response2}")
conversation.append(AIMessage(content=response2))

# Turn 3 — testing memory
conversation.append(HumanMessage(content="Give me a Python code example of what we just discussed."))
response3 = ai.chat(conversation)
print(f"\nAI: {response3}")

AI: An embedding is a way to represent complex data, like words or images, as dense vectors in a high-dimensional space. This allows similar items to be mapped close together, enabling efficient comparison and analysis.

AI: In Retrieval-Augmented Generation (RAG) models, embeddings are used to represent both the input query and the knowledge base. The model uses these embeddings to retrieve relevant information from the knowledge base, and then generates text based on the retrieved information and the input query.

AI: ```python
# Import necessary libraries
import torch
from transformers import AutoModel, AutoTokenizer
import numpy as np

# Load pre-trained model and tokenizer
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define a function to get embeddings
def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model(**inputs)
    embeddings 

**Key insight:** The AI doesn't actually "remember" anything. We're sending the **entire conversation history** with every API call. This is expensive (more tokens = more cost) — which is why we'll build smarter memory systems on Day 3.

---

## 5. 🏆 Challenge — Build your first AI Agent with Smolagents

So far, we've been calling LLMs — they answer questions from their training data.

**An agent is different.** An agent can:
- 🔍 **Search the web** for current information
- 🛠️ **Use tools** (calculator, file reader, API calls)
- 🧠 **Reason** about what tool to use and when
- 🔄 **Iterate** — if the first result isn't good enough, try again

Let's build one in **3 lines** using Hugging Face's Smolagents:

In [12]:
# ============================================================
# COLAB USERS ONLY — Uncomment and run this cell before running agent code below
# Local VS Code users: SKIP this cell (you already have these packages)
# ============================================================

# !pip install "smolagents[litellm]" ddgs -q

In [13]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, InferenceClientModel

# Create an agent with web search capability — 3 lines!
agent = CodeAgent(
    tools=[DuckDuckGoSearchTool()],
    model=InferenceClientModel()
)

# Ask it something that requires CURRENT information (not in training data)
result = agent.run("What are the latest developments in Agentic AI in March 2026?")
print(result)

# NOTE: If you get a 402 "credits depleted" error, HuggingFace free credits ran out.
# Skip to Section 5.3 below to use Groq instead — it has much more generous free limits.

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What are the latest developments in Agentic AI in March 2026?                                                   │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  results = web_search("latest developments in Agentic AI March 2026")                                             
  print(results)                                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[Agentic AI News Roundup (7–13 Mar 2026): Key 
Trends](https://bostoninstituteofanalytics.org/blog/agentic-ai-news-roundup-7-13-march-2026-market-growth-enterpris
e-adoption-new-ai-agents/)
4 days ago -The new market analysis reports which became available between March 7 and 13 now show thatthe global 
agentic AI sector will expand from its current value of $9.14 billion in early 2026 to reach more than $139 billion
by 2034.

[Latest AI & Technology News Roundup – March 
2026](https://www.vtnetzwelt.com/ai-development/latest-ai-technology-news-roundup-march-2026/)
1 day ago -The AI breakthroughs march 2026 wave comprises a generational leap in agentic workflows along with1M 
token context models, serious hardware momentum, and increased national investment in AI infrastructure worldwide 
for the US.

[Alibaba launches agentic AI tool for businesses with Slack, Teams integration 
plans](https://www.cnbc.com/2026/03/17/alibaba-wukong-ai-enterprise-tool-restructuring-qwen-exits.html)
1 day ago -Alibaba is the latest company to roll out AI agents. Rival Tencent and startups such as Zhipu AI have 
raced to launch similar products built on OpenClaw, an open-source agentic platform developed by Peter Steinberger,
who has since joined Sam ...

[How agentic AI will reshape engineering workflows in 2026 | 
CIO](https://www.cio.com/article/4134741/how-agentic-ai-will-reshape-engineering-workflows-in-2026.html)
1 month ago -The most immediate and tangible impact will be on development velocity. We are moving beyond AI as a 
sophisticated coding assistant to AI as an autonomous, multi-skilled team member. Agentic AI will increasingly act 
as a first-pass executor across the SDLC, analyzing feasibility during planning, implementing features during 
build, expanding test coverage during validation and surfacing risks during review; compressing weeks of 
coordination into continuous workflows.

[March 12, 2026 AI News | Latest Artificial Intelligence Updates | 
AIToolly](https://aitoolly.com/ai-news/2026-03-12)
1 week ago -Superpowers, an agentic skills framework and software development methodology, has been introduced as a
complete workflow for coding agents. Built upon a foundation of composable 'skills,' this new approach aims to 
provide an effective solution ...

[Why the Next Era of Agentic Automation Changes 
Everything](https://beam.ai/agentic-insights/ai-landscape-2026-why-the-era-of-agentic-automation-changes-everything
)
2 weeks ago -Conversational interfaces powered by AI are reshaping customer service interactions in 2026. These 
systems handle routine inquiries, provide instant responses, and free human agents to focus on complex customer 
interactions.

[Under the Hood of Agentic AI: Attacks and Defenses in 2026 | by SR | Tech Strories | Mar, 2026 | 
Medium](https://medium.com/@sreekant1729/under-the-hood-of-agentic-ai-attacks-and-defenses-in-2026-0c1f01522910)
3 days ago -These findings highlight the importance of strong governance and security controls for AI agents. As 
threats evolve, researchers and organizations are developing new strategies to secure agentic systems.

[Agentic AI strategy | Deloitte 
Insights](https://www.deloitte.com/us/en/insights/topics/technology-management/tech-trends/2026/agentic-ai-strategy
.html)
10 December 2025 -Agentic AI isn’t enough. Toyota’s Jason Ballard shares how process redesign and people are key to
driving competitive advantage in their transformation. ... Deloitte predicts 2026 will see the gap between the 
promise and reality of AI narrow, as further movements towards getting it to scale are made

[5 Key Trends Shaping Agentic Development in 2026 - The New 
Stack](https://thenewstack.io/5-key-trends-shaping-agentic-development-in-2026/)
27 December 2025 -Five trends in AI development for 2026, includingMCP management, parallel running, CLI vs. 
desktop tools, and challenges with VS Code forks.

[The Agentic AI Revolution: How 2026 Will Reshape Te

[Step 1: Duration 14.57 seconds| Input tokens: 2,092 | Output tokens: 1,329]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("In March 2026, key Agentic AI developments include rapid market growth (projected $139B by 2034),  
  enterprise integration (Alibaba's Slack/Teams tools), autonomous software development workflows, 1M token        
  context models, US Genesis Mission for scientific breakthroughs, Chinese cyberattack automation, and enhanced    
  security governance frameworks.")                                                                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: In March 2026, key Agentic AI developments include rapid market growth (projected $139B by 2034), 
enterprise integration (Alibaba's Slack/Teams tools), autonomous software development workflows, 1M token context 
models, US Genesis Mission for scientific breakthroughs, Chinese cyberattack automation, and enhanced security 
governance frameworks.

[Step 2: Duration 18.83 seconds| Input tokens: 5,498 | Output tokens: 3,617]

In March 2026, key Agentic AI developments include rapid market growth (projected $139B by 2034), enterprise integration (Alibaba's Slack/Teams tools), autonomous software development workflows, 1M token context models, US Genesis Mission for scientific breakthroughs, Chinese cyberattack automation, and enhanced security governance frameworks.


**Watch what happened:**
1. The agent received your question
2. It **decided** it needs to search the web (not just answer from memory)
3. It **wrote Python code** to call the search tool
4. It **analyzed** the search results
5. It **synthesized** a final answer

This is the **ReAct pattern** (Reason + Act) — the foundation of all AI agents. You'll master this pattern over the next 17 days.

### 5.1 Let's add more tools — a multi-tool agent

In [14]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, LiteLLMModel, tool
import os
import litellm
litellm.suppress_debug_info = True

# Create custom tools using the @tool decorator
@tool
def calculate(expression: str) -> str:
    """Evaluates a mathematical expression and returns the result.
    
    Args:
        expression: A mathematical expression like '2 + 2' or '(15 * 3) / 4.5'
    """
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

@tool
def get_word_count(text: str) -> str:
    """Counts the number of words, characters, and sentences in the given text.
    
    Args:
        text: The text to analyze
    """
    words = len(text.split())
    chars = len(text)
    sentences = text.count('.') + text.count('!') + text.count('?')
    return f"Words: {words}, Characters: {chars}, Sentences: {sentences}"

# Using Groq with 8B model — faster and higher token limits for agent workflows
# Note: Agents make 3-5+ LLM calls per query, so they eat tokens fast!
# The 8B model is ideal for agent exercises. Use 70B for single LLM calls.
groq_model = LiteLLMModel(
    model_id="groq/llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY")
)

agent = CodeAgent(
    tools=[DuckDuckGoSearchTool(), calculate, get_word_count],
    model=groq_model
)

# Query 1: The agent should use web search
print("--- Query 1: Needs web search ---")
result1 = agent.run("Who won the latest cricket match between India and Australia?")
print(result1)

--- Query 1: Needs web search ---


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Who won the latest cricket match between India and Australia?                                                   │
│                                                                                                                 │
╰─ LiteLLMModel - groq/llama-3.1-8b-instant ──────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  latest_match = web_search(query="latest India vs Australia cricket match result")                                
  print(latest_match)                                                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[T20 World Cup 2026 | Live Cricket Scores, Today's Match Scorecard 
...](https://www.espncricinfo.com/live-cricket-score)
Catch the fastest livecricketscores and ball-by-ball commentary. Get real-time updates, full scorecards, 
andmatchanalysis for the T20 World Cup 2026 & all global and domestic series.

[India vs Australia Live: IND vs AUS Cricket Score & Ball-by-Ball 
...](https://www.news18.com/cricket/live-score/india-vs-australia-ball-by-ball-live-commentary-auin11082025258933.h
tml)
IndiavsAustraliaLive Score, 5th T20IIndiainAustralia5 T20I Series 2025: Get livecricketscore updates and 
ball-by-ball commentary of INDvsAUS at The Gabba, Brisbane. Stay updated with 
theIndiavsAustralia2025latestmatchdetails, ball by ball updates and full scorecard only on News18.

[India Vs Australia Women Final Live Score - Times 
Now](https://www.timesnownews.com/sports/cricket/india-women-vs-australia-women-live-score-ind-vs-aus-final-t20-tod
ay-scorecard-updates-liveblog-153660100)
IndiaWomenvsAustraliaWomen Live Score: The series is tied at 1-1, andIndiaWomen have a great chance to pull off a 
rare series inAustralia. Can they stun the hosts? Stay tuned for live score updates.,CricketNews, Times Now

[India vs Australia, Live Score, 3rd ODI 
Match,](https://www.india.com/cricket/series/india-in-australia-2025-209183/live-score-3rd-odi-match/india-vs-austr
alia/full-scorecard/258925/)
IndiainAustralia, 2025 Live Score - Full scorecard,IndiainAustralia, 2025cricketscore and updates. Get 
alllatestcricketmatchresults, scores and statistics, with completecricket...

[India Tour of Australia 2025 - IND vs AUS Latest News & Match 
Updates](https://www.sportskeeda.com/go/india-vs-australia)
INDvsAUS - Get all thelatestIndiaTour ofAustralianews, score, livematchupdates, and ball by ball Commentary. Follow
Sportskeeda for all thelatestIndiaTour ofAustralia2025results...

[India vs Australia Highlights, 5th T20I: Match Abandoned Due To Rain 
...](https://sports.ndtv.com/australia-vs-india-2025/india-vs-australia-5th-t20i-live-score-india-tour-of-australia
-2025-ind-vs-aus-live-cricket-scorecard-updates-suryakumar-yadav-abhishek-sharma-shubman-9597850)
IndiavsAustraliaCricketUpdates, 5th T20I: Lightning and rain halted play for over two hours, after which the fifth 
T20I betweenIndiaandAustraliawas abandoned.

[India vs Australia Live Match Scorecard, India vs Australia Cricket 
...](https://www.abplive.com/sports/cricket/live-score/fullscorecard/india-vs-australia-63689c)
MatchInfoMatch:IndiavsAustralia, Semi Final 1, ICC Men's Champions Trophy 2025 TOSS:Australiawon the toss and 
elected to bat.

[Cricket commentary | Australia vs India, 2nd ODI, India tour of 
...](https://www.cricbuzz.com/live-cricket-scores/116921/aus-vs-ind-2nd-odi-india-tour-of-australia-2025)
Follow AUS 265/8 (46.2)vsIND 264/9 (Cooper Connolly 61(53) Adam Zampa 0(1)) |AustraliavsIndia, 2nd ODI,Indiatour 
ofAustralia, 2025, Thu, Oct 23,Indiatour ofAustralia, 2025 with live ...

[Australia vs India Highlights, 3rd ODI at Sydney: Rohit, Kohli steer 
...](https://www.firstpost.com/firstcricket/india-vs-australia-3rd-odi-live-score-ind-vs-aus-full-scorecard-sydney-
cricket-ground-kohli-liveblog-13944886.html)
IndiavsAustralia3rd ODI LiveCricketScore UpdatesIndiavsAustraliaLive Score, 3rd ODI: 
Catchlatestupdates,matchcommentary and full scorecard from the INDvsAUS 3rd ODI in Sydney on Saturday (25 October).

[Australia v India: third women's T20 international - The 
Guardian](https://www.theguardian.com/sport/live/2026/feb/21/australia-vs-india-third-womens-t20-international-cric
ket-aus-v-ind-live-updates-scores)
Phoebe Litchfield departs during the third women's T20I of the multi-format series at Adelaide Oval. Follow for 
live scores andcricketupdates from AusvsInd. Photograph: Matt Turner/AAP

Out: None

...

[Step 4: Duration 204.58 seconds| Input tokens: 15,971 | Output tokens: 1,450]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Request too large for model 
`llama-3.1-8b-instant` in organization `org_01kfzzwk76f0y83hedfy0vefyv` service tier `on_demand` on tokens per 
minute (TPM): Limit 6000, Requested 6990, please reduce your message size and try again. Need more tokens? Upgrade 
to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

[Step 5: Duration 498.11 seconds]

AgentGenerationError: Error in generating model output:
litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Request too large for model `llama-3.1-8b-instant` in organization `org_01kfzzwk76f0y83hedfy0vefyv` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 6990, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}


In [ ]:
# Query 2: The agent should use the calculate tool (reuses agent from above)
print("--- Query 2: Needs calculator ---")
result2 = agent.run("What is 245 multiplied by 38, plus 127?")
print(result2)
print()

# Query 3: The agent should use the word count tool
print("--- Query 3: Needs word count ---")
result3 = agent.run("Count the words in this text: Agentic AI is the future of software engineering and it will transform how we build applications")
print(result3)

### 5.2 🎯 Your turn — Build a custom agent

**Challenge (10 minutes):** Create an agent with at least one custom tool of your choice. Ideas:
- A tool that converts temperatures (Celsius ↔ Fahrenheit)
- A tool that generates a random motivational quote
- A tool that checks if a number is prime
- A tool that reverses a string

Then test your agent with 3 different queries — make sure it picks the right tool each time.

In [ ]:
# YOUR CODE HERE
# Step 1: Create a custom tool using @tool decorator
# Step 2: Create a CodeAgent with your tool + DuckDuckGoSearchTool
# Step 3: Test with 3 queries — at least one should use your custom tool

# Example skeleton:
# from smolagents import CodeAgent, DuckDuckGoSearchTool, LiteLLMModel, tool
# import os
#
# @tool
# def my_custom_tool(input: str) -> str:
#     """Description of what your tool does.
#     
#     Args:
#         input: What the input represents
#     """
#     # Your logic here
#     return result
#
# groq_model = LiteLLMModel(
#     model_id="groq/llama-3.1-8b-instant",
#     api_key=os.getenv("GROQ_API_KEY")
# )
#
# agent = CodeAgent(
#     tools=[DuckDuckGoSearchTool(), my_custom_tool],
#     model=groq_model
# )
#
# print(agent.run("Your test query here"))

### 5.3 HuggingFace vs Groq — choosing the right backend

By default, Smolagents uses HuggingFace Inference. But HuggingFace free credits run out fast with agents (each run makes 3-5+ LLM calls). **Groq is the better choice for agent work** — faster, more generous free tier.

We already switched to Groq in Section 5.1 above. Here's how both options compare:

| Backend | Speed | Free Limits | Best For |
|---------|-------|-------------|----------|
| HuggingFace (`InferenceClientModel`) | Moderate | Limited monthly credits | Quick single demos |
| Groq (`LiteLLMModel`) | Very fast (300+ tok/sec) | ~1000 req/day per account | Agent workflows, course exercises |

In [ ]:
# You can also switch back to HuggingFace if you have credits remaining
from smolagents import CodeAgent, DuckDuckGoSearchTool, InferenceClientModel, LiteLLMModel
import os

# Compare the two backends:
# Option A: HuggingFace (free but limited credits — runs out fast with agents)
# hf_model = InferenceClientModel()

# Option B: Groq (free, fast, generous limits — recommended for this course)
groq_model = LiteLLMModel(
    model_id="groq/llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY")
)

agent = CodeAgent(
    tools=[DuckDuckGoSearchTool()],
    model=groq_model
)

result = agent.run("What is the population of Visakhapatnam?")
print(result)

---
## 6. Day 1 Wrap-up

### What you built today:
- ✅ Called LLM APIs via raw HTTP and LangChain
- ✅ Understood tokens, temperature, system prompts, and streaming
- ✅ Built a reusable `AgentLLM` wrapper class (you'll use this throughout the course)
- ✅ Built your first AI agent with Smolagents — web search, custom tools, reasoning
- ✅ Saw the ReAct pattern in action (Reason → Act → Observe → Respond)

### Key files to save:
- This notebook (your reference for API patterns)
- The `AgentLLM` class — we'll extend it throughout the course

### Tomorrow — Day 2: Tool Calling & Function Agents
We'll go deeper into **function calling** — the mechanism that lets LLMs decide which tool to use. You'll build agents with 5+ tools that can chain actions together.

---

### 📚 Want to explore more tonight?
- [Groq API docs](https://console.groq.com/docs/quickstart)
- [LangChain Groq integration](https://python.langchain.com/docs/integrations/chat/groq/)
- [Smolagents docs](https://huggingface.co/docs/smolagents)
- [HuggingFace Agents Course](https://huggingface.co/learn/agents-course)